# 02. 单卡 LoRA 微调

本 Notebook 运行 5 step 训练，检查数据编码、adapter 保存和结果记录。完整训练参数见 `docs/zh/02_finetune/`。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "cases").exists() and (REPO_ROOT.parent / "cases").exists():
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = Path("/mnt/workspace/data/Qwen2.5-0.5B-Instruct")
DATA_FILE = REPO_ROOT / "cases/qwen/datasets/huanhuan-100.json"
ADAPTER_DIR = REPO_ROOT / "cases/qwen/results/notebook-lora-smoke"
MERGED_DIR = REPO_ROOT / "cases/qwen/results/notebook-lora-merged"

assert (MODEL_PATH / "config.json").exists(), f"找不到模型：{MODEL_PATH}"
assert DATA_FILE.exists(), f"找不到数据：{DATA_FILE}"
print("repo:", REPO_ROOT)
print("model:", MODEL_PATH)
print("data:", DATA_FILE)

In [ ]:
import torch
import torch_npu
import transformers
import peft
import datasets

print("torch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
assert torch.npu.is_available()

In [ ]:
sys.path.insert(0, str(REPO_ROOT))
from transformers import AutoTokenizer
from cases.qwen.scripts.run_lora_sft import build_dataset, load_records

records = load_records(str(DATA_FILE))
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False, local_files_only=True)
train_dataset = build_dataset(
    tokenizer,
    str(DATA_FILE),
    "现在你要扮演皇帝身边的女人--甄嬛。",
    128,
)
sample = train_dataset[0]
print("rows:", len(records))
print("tokens:", len(sample["input_ids"]))
print("supervised tokens:", sum(value != -100 for value in sample["labels"]))

In [ ]:
command = [
    sys.executable,
    str(REPO_ROOT / "cases/qwen/scripts/run_lora_sft.py"),
    "--model", str(MODEL_PATH),
    "--local-files-only",
    "--data-file", str(DATA_FILE),
    "--output-dir", str(ADAPTER_DIR),
    "--max-steps", "5",
    "--per-device-train-batch-size", "1",
    "--gradient-accumulation-steps", "1",
    "--max-length", "128",
    "--logging-steps", "1",
    "--save-steps", "1000",
    "--no-gradient-checkpointing",
]
subprocess.run(command, cwd=REPO_ROOT, check=True)

In [ ]:
result_file = max(
    (REPO_ROOT / "cases/qwen/results").glob("lora_sft_*.json"),
    key=lambda p: p.stat().st_mtime,
)
result = json.loads(result_file.read_text(encoding="utf-8"))
print("lora:", result["lora"])
print("training:", result["training"])
print("metrics:", result["metrics"])
print("result_file:", result_file)

下面的单元会生成一份完整合并模型，占用空间明显大于 adapter。确认磁盘空间后，把 `RUN_MERGE` 改为 `True`。

In [ ]:
RUN_MERGE = False

if RUN_MERGE:
    merge_command = [
        sys.executable,
        str(REPO_ROOT / "cases/qwen/scripts/merge_lora.py"),
        "--base-model", str(MODEL_PATH),
        "--local-files-only",
        "--adapter-path", str(ADAPTER_DIR),
        "--output-dir", str(MERGED_DIR),
        "--verify-prompt", "你是谁？",
        "--max-new-tokens", "32",
    ]
    subprocess.run(merge_command, cwd=REPO_ROOT, check=True)
else:
    print("未执行合并。将 RUN_MERGE 改为 True 后重新运行本单元。")